# ADIRE Simulation Results

Analysis of the ADIRE (Anchor-Diffed Incremental Re-Embedding) simulation comparing five re-embedding strategies across document sizes, structures, edit types, and edit chains.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

from adire.analysis import (
    SAME_GRANULARITY_STRATEGIES,
    STRATEGY_LABELS,
    STRATEGY_ORDER,
    chain_results,
    compute_cost_savings,
    format_size_label,
    heatmap_data,
    load_results,
    mean_by,
    pivot_for_grouped_bar,
    single_edit_results,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

df = load_results("../results.parquet")
single = single_edit_results(df)
chains = chain_results(df)

print(f"Total rows: {len(df):,}")
print(f"Single-edit: {len(single):,}")
print(f"Chain: {len(chains):,}")

## Data Validation

Verify the dataset is complete and well-formed before analysis.

In [ ]:
# --- No missing values ---
assert df.isna().sum().sum() == 0, f"Missing values found:\n{df.isna().sum()[df.isna().sum() > 0]}"

# --- Expected experiment types ---
assert set(df["experiment_type"].unique()) == {"single", "chain"}

# --- All 5 strategies present ---
expected_strategies = {"naive", "paragraph_reuse", "chunk_hash", "adire", "adire_wide_window"}
assert set(df["strategy"].unique()) == expected_strategies

# --- Metrics are in valid ranges ---
for col in ["preservation_rate", "reembedding_rate", "token_savings_rate", "fragment_ratio", "oversized_ratio"]:
    assert (df[col] >= 0.0).all(), f"{col} has negative values"
    assert (df[col] <= 1.0).all(), f"{col} has values > 1.0"

assert (df["chunks_reembedded"] >= 0).all()
assert (df["chunks_reused"] >= 0).all()
assert (df["tokens_reembedded"] >= 0).all()
assert (df["algorithm_time_ms"] >= 0).all()

# --- Chunk counts are consistent ---
assert (df["chunks_reembedded"] + df["chunks_reused"] == df["chunk_count_after"]).all(), \
    "chunks_reembedded + chunks_reused != chunk_count_after"

# --- Naive always has 0% preservation ---
naive = df[df["strategy"] == "naive"]
assert (naive["chunks_reused"] == 0).all(), "Naive strategy should never reuse chunks"
assert (naive["preservation_rate"] == 0.0).all()

# --- Row counts ---
print(f"Single-edit rows: {len(single):,}")
print(f"Chain rows: {len(chains):,}")
print(f"Strategies: {sorted(df['strategy'].unique())}")
print(f"Document sizes: {sorted(df['document_size'].unique())}")
print(f"Document profiles: {sorted(df['document_profile'].unique())}")
print(f"Edit types: {sorted(df['edit_type'].unique())}")
print(f"\nAll validation checks passed.")

## 1. Token Savings Rate by Document Size

The headline chart. `token_savings_rate` is comparable across all strategies regardless of chunking granularity because it's denominated in tokens, not chunks. Shows where ADIRE starts to matter.

In [ ]:
pivoted = pivot_for_grouped_bar(single, "document_size", "token_savings_rate")
ax = pivoted.rename(columns=STRATEGY_LABELS).plot(kind="bar", width=0.8)
ax.set_xlabel("Document Size (chars)")
ax.set_ylabel("Token Savings Rate")
ax.set_title("Token Savings Rate by Document Size")
ax.set_xticklabels([format_size_label(x) for x in pivoted.index], rotation=0)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../notebooks/fig01_token_savings_by_size.png", bbox_inches="tight")
plt.show()

## 2. Preservation Rate by Document Size (Same-Granularity Strategies)

Apples-to-apples comparison of strategies with the same chunking granularity (greedy combining to ~512 tokens). Paragraph-level reuse is excluded because its chunks are individual paragraphs — its preservation rate measures a different thing.

In [ ]:
pivoted = pivot_for_grouped_bar(
    single, "document_size", "preservation_rate",
    strategies=SAME_GRANULARITY_STRATEGIES,
)
ax = pivoted.rename(columns=STRATEGY_LABELS).plot(kind="bar", width=0.8)
ax.set_xlabel("Document Size (chars)")
ax.set_ylabel("Preservation Rate")
ax.set_title("Preservation Rate by Document Size (Same-Granularity Strategies)")
ax.set_xticklabels([format_size_label(x) for x in pivoted.index], rotation=0)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../notebooks/fig02_preservation_by_size.png", bbox_inches="tight")
plt.show()

## 3. Preservation Rate by Document Structure

Shows which document structures benefit most from ADIRE. The structureless blob is the degenerate case — with few paragraph boundaries, ADIRE has limited granularity for change detection.

In [ ]:
pivoted = pivot_for_grouped_bar(
    single, "document_profile", "preservation_rate",
    strategies=SAME_GRANULARITY_STRATEGIES,
)
ax = pivoted.rename(columns=STRATEGY_LABELS).plot(kind="bar", width=0.8)
ax.set_xlabel("Document Profile")
ax.set_ylabel("Preservation Rate")
ax.set_title("Preservation Rate by Document Structure")
ax.set_xticklabels(pivoted.index, rotation=30, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../notebooks/fig03_preservation_by_structure.png", bbox_inches="tight")
plt.show()

## 4. Preservation Rate by Edit Type

Shows which edits benefit most from ADIRE. Hypothesis: typo fix and append are near 100% preservation, section rewrite is lower.

In [ ]:
pivoted = pivot_for_grouped_bar(
    single, "edit_type", "preservation_rate",
    strategies=SAME_GRANULARITY_STRATEGIES,
)
ax = pivoted.rename(columns=STRATEGY_LABELS).plot(kind="bar", width=0.8)
ax.set_xlabel("Edit Type")
ax.set_ylabel("Preservation Rate")
ax.set_title("Preservation Rate by Edit Type")
ax.set_xticklabels(pivoted.index, rotation=30, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../notebooks/fig04_preservation_by_edit_type.png", bbox_inches="tight")
plt.show()

## 5. Token Savings Heatmap (Document Size x Edit Type)

The primary result visualization — shows the full interaction between document size and edit type for ADIRE.

In [ ]:
hm = heatmap_data(single, "document_size", "edit_type", "token_savings_rate", "adire")
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(hm.values, cmap="YlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(hm.columns)))
ax.set_xticklabels(hm.columns, rotation=30, ha="right")
ax.set_yticks(range(len(hm.index)))
ax.set_yticklabels([format_size_label(x) for x in hm.index])
ax.set_xlabel("Edit Type")
ax.set_ylabel("Document Size")
ax.set_title("ADIRE Token Savings Rate (Document Size x Edit Type)")
# Annotate cells
for i in range(len(hm.index)):
    for j in range(len(hm.columns)):
        val = hm.values[i, j]
        color = "white" if val > 0.7 else "black"
        ax.text(j, i, f"{val:.0%}", ha="center", va="center", color=color, fontsize=9)
fig.colorbar(im, ax=ax, label="Token Savings Rate", format=mtick.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig("../notebooks/fig05_token_savings_heatmap.png", bbox_inches="tight")
plt.show()

## 6. Cascade Effect: Chunk-Hash Match vs ADIRE by Edit Position

Demonstrates the cascade problem. Chunk-hash match preservation should drop for top-of-document edits (boundaries shift downstream), while ADIRE remains stable regardless of edit position.

In [ ]:
cascade_strategies = ["chunk_hash", "adire"]
cascade_df = single[single["strategy"].isin(cascade_strategies)]
# Exclude position-independent edit types
cascade_df = cascade_df[~cascade_df["edit_type"].isin(["append", "scattered_edits"])]

pivoted = pivot_for_grouped_bar(
    cascade_df, "edit_position", "preservation_rate",
    strategies=cascade_strategies,
)
ax = pivoted.rename(columns=STRATEGY_LABELS).plot(kind="bar", width=0.6)
ax.set_xlabel("Edit Position")
ax.set_ylabel("Preservation Rate")
ax.set_title("Cascade Effect: Chunk-Hash Match vs ADIRE by Edit Position")
ax.set_xticklabels(pivoted.index, rotation=0)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.savefig("../notebooks/fig06_cascade_by_position.png", bbox_inches="tight")
plt.show()

## 7. Fragmentation Over Edit Chains

Shows whether fragmentation accumulates over sequential edits. The defrag threshold (30% fragment ratio) is shown as a dashed line.

In [ ]:
chain_strategies = ["adire", "adire_wide_window"]
chain_frag = chains[chains["strategy"].isin(chain_strategies)]
frag_by_step = chain_frag.groupby(["trial_number", "strategy"])["fragment_ratio"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
for strategy in chain_strategies:
    data = frag_by_step[frag_by_step["strategy"] == strategy]
    ax.plot(data["trial_number"], data["fragment_ratio"],
            marker="o", label=STRATEGY_LABELS[strategy], markersize=4)
ax.axhline(y=0.30, color="red", linestyle="--", alpha=0.7, label="Defrag threshold (30%)")
ax.set_xlabel("Chain Step")
ax.set_ylabel("Fragment Ratio")
ax.set_title("Fragmentation Over Edit Chains")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.savefig("../notebooks/fig07_fragmentation_chains.png", bbox_inches="tight")
plt.show()

## 8. Chunk Size Distribution: ADIRE vs From-Scratch (Naive)

Compares chunk quality between ADIRE's incremental re-chunking and naive's from-scratch chunking. Naive always produces optimal chunk sizes; ADIRE's boundary preservation may produce more fragments.

In [ ]:
frag_compare = mean_by(
    single[single["strategy"].isin(["naive", "adire", "adire_wide_window"])],
    ["document_size", "strategy"],
    "fragment_ratio",
)
pivoted = frag_compare.pivot_table(index="document_size", columns="strategy", values="fragment_ratio")
cols = [s for s in STRATEGY_ORDER if s in pivoted.columns]
ax = pivoted[cols].rename(columns=STRATEGY_LABELS).plot(kind="bar", width=0.7)
ax.set_xlabel("Document Size (chars)")
ax.set_ylabel("Fragment Ratio")
ax.set_title("Chunk Quality: Fragment Ratio by Strategy and Document Size")
ax.set_xticklabels([format_size_label(x) for x in pivoted.index], rotation=0)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../notebooks/fig08_chunk_quality.png", bbox_inches="tight")
plt.show()

## 9. Cost Savings Estimate

Using approximate embedding costs ($0.00002 per 1K tokens), estimated dollar savings per edit across document sizes.

In [ ]:
cost_df = compute_cost_savings(single)
cost_by_size = cost_df.groupby(["document_size", "strategy"])["cost_savings_usd"].mean().reset_index()
pivoted = cost_by_size.pivot_table(index="document_size", columns="strategy", values="cost_savings_usd")
# Exclude naive (savings are always 0)
cols = [s for s in STRATEGY_ORDER if s in pivoted.columns and s != "naive"]
ax = pivoted[cols].rename(columns=STRATEGY_LABELS).plot(kind="bar", width=0.8)
ax.set_xlabel("Document Size (chars)")
ax.set_ylabel("Cost Savings per Edit (USD)")
ax.set_title("Estimated Cost Savings per Edit vs Naive")
ax.set_xticklabels([format_size_label(x) for x in pivoted.index], rotation=0)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("$%.6f"))
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../notebooks/fig09_cost_savings.png", bbox_inches="tight")
plt.show()

# Print summary table
print("\nMean cost savings per edit (USD) vs naive:")
print(pivoted[cols].rename(columns=STRATEGY_LABELS).to_string(float_format="${:.8f}".format))

## 10. Algorithm Overhead vs API Latency Savings

Shows algorithm CPU time and estimated API latency as components of total latency. The crossover point where ADIRE's CPU overhead is justified by API latency savings.

In [ ]:
latency_strategies = ["naive", "chunk_hash", "adire", "adire_wide_window"]
latency_df = single[single["strategy"].isin(latency_strategies)]

algo_time = mean_by(latency_df, ["document_size", "strategy"], "algorithm_time_ms")
api_time = mean_by(latency_df, ["document_size", "strategy"], "estimated_api_latency_ms")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: algorithm time
algo_pivot = algo_time.pivot_table(index="document_size", columns="strategy", values="algorithm_time_ms")
cols = [s for s in latency_strategies if s in algo_pivot.columns]
algo_pivot[cols].rename(columns=STRATEGY_LABELS).plot(kind="bar", ax=axes[0], width=0.8)
axes[0].set_xlabel("Document Size")
axes[0].set_ylabel("Algorithm Time (ms)")
axes[0].set_title("Algorithm CPU Time")
axes[0].set_xticklabels([format_size_label(x) for x in algo_pivot.index], rotation=0)
axes[0].legend(fontsize=8)

# Right: estimated API latency
api_pivot = api_time.pivot_table(index="document_size", columns="strategy", values="estimated_api_latency_ms")
api_pivot[cols].rename(columns=STRATEGY_LABELS).plot(kind="bar", ax=axes[1], width=0.8)
axes[1].set_xlabel("Document Size")
axes[1].set_ylabel("Estimated API Latency (ms)")
axes[1].set_title("Estimated Embedding API Latency")
axes[1].set_xticklabels([format_size_label(x) for x in api_pivot.index], rotation=0)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../notebooks/fig10_latency_components.png", bbox_inches="tight")
plt.show()

## 11. Estimated Total Latency by Document Size

End-to-end latency comparison. Shows at what document size ADIRE becomes faster than naive, accounting for both algorithm CPU cost and embedding API calls.

In [ ]:
total_latency = mean_by(latency_df, ["document_size", "strategy"], "estimated_total_latency_ms")

fig, ax = plt.subplots(figsize=(10, 6))
for strategy in latency_strategies:
    data = total_latency[total_latency["strategy"] == strategy].sort_values("document_size")
    ax.plot(data["document_size"], data["estimated_total_latency_ms"],
            marker="o", label=STRATEGY_LABELS[strategy], markersize=6)
ax.set_xlabel("Document Size (chars)")
ax.set_ylabel("Estimated Total Latency (ms)")
ax.set_title("Estimated Total Latency by Document Size")
ax.set_xticks(sorted(latency_df["document_size"].unique()))
ax.set_xticklabels([format_size_label(x) for x in sorted(latency_df["document_size"].unique())])
ax.legend()
plt.tight_layout()
plt.savefig("../notebooks/fig11_total_latency.png", bbox_inches="tight")
plt.show()

## Summary

Key findings from the simulation:

In [ ]:
adire_single = single[single["strategy"] == "adire"]
chunk_hash_single = single[single["strategy"] == "chunk_hash"]
para_reuse_single = single[single["strategy"] == "paragraph_reuse"]

print("=" * 70)
print("ADIRE SIMULATION RESULTS SUMMARY")
print("=" * 70)

print("\n--- Token Savings Rate (mean across all edits) ---")
for size in sorted(single["document_size"].unique()):
    adire_savings = adire_single[adire_single["document_size"] == size]["token_savings_rate"].mean()
    ch_savings = chunk_hash_single[chunk_hash_single["document_size"] == size]["token_savings_rate"].mean()
    pr_savings = para_reuse_single[para_reuse_single["document_size"] == size]["token_savings_rate"].mean()
    print(f"  {format_size_label(size):>5s}: ADIRE={adire_savings:.1%}  Chunk-Hash={ch_savings:.1%}  Para-Reuse={pr_savings:.1%}")

print("\n--- Cascade Effect (preservation rate by position) ---")
pos_dep = single[~single["edit_type"].isin(["append", "scattered_edits"])]
for pos in ["top", "middle", "bottom"]:
    adire_rate = pos_dep[(pos_dep["strategy"] == "adire") & (pos_dep["edit_position"] == pos)]["preservation_rate"].mean()
    ch_rate = pos_dep[(pos_dep["strategy"] == "chunk_hash") & (pos_dep["edit_position"] == pos)]["preservation_rate"].mean()
    print(f"  {pos:>6s}: ADIRE={adire_rate:.1%}  Chunk-Hash={ch_rate:.1%}  (delta={adire_rate - ch_rate:+.1%})")

print("\n--- Chain Fragmentation (ADIRE, mean fragment_ratio) ---")
adire_chain = chains[chains["strategy"] == "adire"]
first_step = adire_chain[adire_chain["trial_number"] == 0]["fragment_ratio"].mean()
last_step = adire_chain[adire_chain["trial_number"] == adire_chain["trial_number"].max()]["fragment_ratio"].mean()
print(f"  Step 0: {first_step:.1%}")
print(f"  Step {int(adire_chain['trial_number'].max())}: {last_step:.1%}")
print(f"  Defrag threshold (30%): {'EXCEEDED' if last_step > 0.30 else 'NOT exceeded'}")

print("\n--- Key Takeaways ---")
large_savings = adire_single[adire_single["document_size"] >= 50000]["token_savings_rate"].mean()
print(f"  ADIRE saves {large_savings:.0%} of embedding tokens for large documents (50K+ chars)")
typo_savings = adire_single[adire_single["edit_type"] == "typo_fix"]["token_savings_rate"].mean()
print(f"  Typo fixes (most common edit) save {typo_savings:.0%} of tokens with ADIRE")
print("=" * 70)